# Error Handling

Various errors may occur when communicating with the IM7.

There is a version of the communication class that works with exceptions [ZahnerLinkExc](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc), and a version that works without exceptions [ZahnerLink](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLink), requiring you to check the return value manually.

The other examples do not consider error handling, as it is usually the user's responsibility to decide how to handle and resolve errors. This also depends on the application.

In [2]:
import os
import sys
import threading
import subprocess
import time
import zahner_link as zl

def print_error_object(error_object: zl.ErrorObject):
    print(f"ErrorObject: {error_object}")
    print(f"ErrorObject==True: {error_object==True}")
    print(f"ErrorObject code enum: {error_object.get_error_code_enum()}")
    print(f"ErrorObject message: {error_object.get_message_formatted()}")

def print_job_status(job: zl.AbstractMeasurementJob):
    print(f"job successfull: {job.was_successful()}")
    print(f"job status: {job.get_last_job_status()}")
    print(f"job error message: {job.get_last_job_error_message()}")
    info = job.get_last_job_info()
    print(f"JobInfo object id: {info.get_job_id()}")
    print(f"JobInfo object status: {info.get_status()}")
    print(f"JobInfo object status detail: {info.get_status_detail()}")
    print(f"JobInfo object error message: {info.get_error_message()}")
    print(f"JobInfo started: {info.get_start_date()}")

HOST = "10.10.253.150"
PORT = "1994"

## Runtime Error

Runtime errors may occur due to incorrect operation of the IM7.

### Exception Library

First, the exception-based library is used to explain the error output when a job fails.

The two helper functions `print_error_object` and `print_job_status` are used for this purpose.

In [3]:
link = zl.ZahnerLinkExc(HOST, PORT)
error: zl.ErrorObject = link.connect()

if not error:
    print("connected successfully")
else:
    print(f"failed to connect, status: {error.get_error_code_enum()}, message: {error.get_message_formatted()}")

switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")

try:
    print("\nfirst switch on")
    error_object_success: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_success)

    print("\nsecond switch on")
    error_object_fail: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_fail)

except zl.ZahnerLinkException as e:
    print(f"\ncaught exception: {e}")
    error_object: zl.ErrorObject = e.error
    print_error_object(error_object)
    print_job_status(switch_on_job)

link.do_job(switch_off_job)
link.disconnect()

connected successfully

first switch on
ErrorObject: no error occoured
ErrorObject==True: False
ErrorObject code enum: ErrorCodeEnum.NONE
ErrorObject message: no error occoured

second switch on

caught exception: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.
ErrorObject: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.UNEXPECTED_EXCEPTION
ErrorObject message: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.
job successfull: False
job status: JobStatusEnum.FAILED
job error message: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.
JobInfo object id: fbc13cec-0954-49fa-93fb-cb0d3901b2e1

### No exception library

As you can see from the following output, the exception was not triggered.

You therefore have to manually check for success in your program flow.

In [4]:
link = zl.ZahnerLink(HOST, PORT)
error: zl.ErrorObject = link.connect()

if not error:
    print("connected successfully")
else:
    print(f"failed to connect, status: {error.get_error_code_enum()}, message: {error.get_message_formatted()}")

switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")

try:
    print("\nfirst switch on")
    error_object_success: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_success)

    print("\nsecond switch on")
    error_object_fail: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_fail)

except zl.ZahnerLinkException as e:
    print(f"\ncaught exception: {e}")
    error_object: zl.ErrorObject = e.error
    print_error_object(error_object)    
    print_job_status(switch_on_job)

link.do_job(switch_off_job)
link.disconnect()

connected successfully

first switch on
ErrorObject: no error occoured
ErrorObject==True: False
ErrorObject code enum: ErrorCodeEnum.NONE
ErrorObject message: no error occoured

second switch on
ErrorObject: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.UNEXPECTED_EXCEPTION
ErrorObject message: an unexpected exception occured: SystemException: type: 8; message: Potentiostat '37128:POT' must be switched off, before switched on.


Another example is when jobs are executed without a connection being established.

In [5]:
link = zl.ZahnerLink("definitely.wrong.hostname", PORT)
error: zl.ErrorObject = link.connect()

if not error:
    print("connected successfully")
else:
    print(f"failed to connect, status: {error.get_error_code_enum()}, message: {error.get_message_formatted()}")

switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")

try:
    print("\nfirst switch on")
    error_object_success: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_success)

    print("\nsecond switch on")
    error_object_fail: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_fail)

except zl.ZahnerLinkException as e:
    print(f"\ncaught exception: {e}")
    error_object: zl.ErrorObject = e.error
    print_error_object(error_object)

link.do_job(switch_off_job)
link.disconnect()

failed to connect, status: ErrorCodeEnum.CONNECTION_DNS_ERROR, message: dns error: definitely.wrong.hostname:1994

first switch on
ErrorObject: no network connection has been established yet
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.CONNECTION_NOT_ESTABLISHED
ErrorObject message: no network connection has been established yet

second switch on
ErrorObject: no network connection has been established yet
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.CONNECTION_NOT_ESTABLISHED
ErrorObject message: no network connection has been established yet


## Parameter Error

If the parameters of a job are incorrect, the library also returns an error message.

Only the exception-based library is shown here.

In [6]:
link = zl.ZahnerLinkExc(HOST, PORT)
error: zl.ErrorObject = link.connect()

if not error:
    print("connected successfully")
else:
    print(f"failed to connect, status: {error.get_error_code_enum()}, message: {error.get_message_formatted()}")

switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")

try:
    error_object_success: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_success)
    
    eis_dict_table_job = zl.meas.EisFrequencyTableJob(
        {
            "bias": 0.0,
            "spectrum": [
                {
                    "frequency": freq,
                    "amplitude": -2, # Negative amplitude to provoke an error
                    "pre_duration": 0.1,
                    "pre_waves": 1,
                    "meas_duration": 0.5,
                    "meas_waves": 3,
                }
                for freq in [1e3,10e3]
            ],
        }
    )
    link.do_job(eis_dict_table_job)

except zl.ZahnerLinkException as e:
    print(f"\ncaught exception: {e}")
    error_object: zl.ErrorObject = e.error
    print_error_object(error_object)
    print_job_status(eis_dict_table_job)

link.do_job(switch_off_job)
link.disconnect()

connected successfully
ErrorObject: no error occoured
ErrorObject==True: False
ErrorObject code enum: ErrorCodeEnum.NONE
ErrorObject message: no error occoured

caught exception: parameter 'amplitude' is too low: '-2' < '0' must be false
ErrorObject: parameter 'amplitude' is too low: '-2' < '0' must be false
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.PARAMETER_VALUE_TOO_LOW
ErrorObject message: parameter 'amplitude' is too low: '-2' < '0' must be false
job successfull: False
job status: JobStatusEnum.FAILED
job error message: parameter 'amplitude' is too low: '-2' < '0' must be false
JobInfo object id: void
JobInfo object status: JobStatusEnum.FAILED
JobInfo object status detail: JobStatusDetailEnum.FAILED_TO_CREATE
JobInfo object error message: parameter 'amplitude' is too low: '-2' < '0' must be false
JobInfo started: 


## Connection Loss

If the connection is lost, you can re-establish the connection to the device and check the job list to see if any active jobs are still running.

In this example, the connection disruption is triggered by the software using PowerShell.

In [ ]:
DELAY_BEFORE_DISCONNECT = 5  # seconds before the connection is killed
ADAPTER_NAME = "Ethernet 2"

def disable_adapter():
    """Disables the network adapter via netsh (requires admin, UAC prompt)."""
    cmd = f'netsh interface set interface "{ADAPTER_NAME}" admin=disable'
    print(f"[NET] Disabling '{ADAPTER_NAME}' ...")
    result = subprocess.run(
        ["powershell", "-Command", f"Start-Process cmd -ArgumentList '/c {cmd}' -Verb RunAs -Wait"],
        capture_output=True, text=True,
    )
    print(f"[NET] rc={result.returncode} stdout={result.stdout.strip()!r} stderr={result.stderr.strip()!r}")

def enable_adapter():
    """Re-enables the network adapter via netsh (requires admin, UAC prompt)."""
    cmd = f'netsh interface set interface "{ADAPTER_NAME}" admin=enable'
    print(f"[NET] Enabling '{ADAPTER_NAME}' ...")
    result = subprocess.run(
        ["powershell", "-Command", f"Start-Process cmd -ArgumentList '/c {cmd}' -Verb RunAs -Wait"],
        capture_output=True, text=True,
    )
    print(f"[NET] rc={result.returncode} stdout={result.stdout.strip()!r} stderr={result.stderr.strip()!r}")

def disconnect_after_delay(delay: float):
    """Waits `delay` seconds and then disables the network adapter."""
    print(f"[DISCONNECT-THREAD] Waiting {delay}s before killing the connection ...")
    time.sleep(delay)
    print(f"[DISCONNECT-THREAD] Disabling adapter NOW!")
    disable_adapter()


link = zl.ZahnerLinkExc(HOST, PORT)
status = link.connect()

if status == zl.ZahnerLinkServiceStatusEnum.SUCCESS_NO_ERROR:
    print("connected successfully")
else:
    print(f"failed to connect, status: {status}")

switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")

eis_dict_table_job = zl.meas.EisFrequencyTableJob(
    {
        "bias": 0.0,
        "spectrum": [
            {
                "frequency": freq,
                "amplitude": 0.1,
                "pre_duration": 0.1,
                "pre_waves": 1,
                "meas_duration": 0.5,
                "meas_waves": 3,
            }
            for freq in [1e3, 10e3, 1, 200e-3]
        ],
    }
)

try:
    error_object_success: zl.ErrorObject = link.do_job(switch_on_job)
    print_error_object(error_object_success)

    # Start disconnect thread — disables adapter after DELAY_BEFORE_DISCONNECT seconds
    disconnect_thread = threading.Thread(
        target=disconnect_after_delay,
        args=(DELAY_BEFORE_DISCONNECT,),
        daemon=True,
    )
    disconnect_thread.start()

    print("\nStarting EIS measurement (connection will be killed mid-measurement) ...")
    link.do_job(eis_dict_table_job)
    print("\nMeasurement unexpectedly completed (connection was not killed fast enough).")

except zl.ZahnerLinkException as e:
    print(f"\ncaught exception: {e}")
    error_object: zl.ErrorObject = e.error
    print_error_object(error_object)
    print_job_status(eis_dict_table_job)

finally:
    # Always re-enable the adapter
    print("\n[CLEANUP] Re-enabling network adapter ...")
    enable_adapter()

# now we are disconnected
print("\nReconnect Now")
link.connect()

running_jobs = 0
job_list = link.get_job_info_list()
for job_info in job_list:
    print(f"id: {job_info.get_job_id()}")
    print(f"\tstatus: {job_info.get_status()}")
    if job_info.get_status() == zl.JobStatusEnum.RUNNING:
        running_jobs += 1
    print(f"\terror message: {job_info.get_error_message()}")
    print(f"\tstarted: {job_info.get_start_date()}")
    print(f"\tended: {job_info.get_end_date()}")
print(f"Number of jobs still running: {running_jobs}")

print("\nWait until EIS is finished")
time.sleep(15)

running_jobs = 0
job_list = link.get_job_info_list()
for job_info in job_list:
    print(f"id: {job_info.get_job_id()}")
    print(f"\tstatus: {job_info.get_status()}")
    if job_info.get_status() == zl.JobStatusEnum.RUNNING:
        running_jobs += 1
    print(f"\terror message: {job_info.get_error_message()}")
    print(f"\tstarted: {job_info.get_start_date()}")
    print(f"\tended: {job_info.get_end_date()}")
print(f"Number of jobs still running: {running_jobs}")

# Switch off the potentiostat
try:
    print("\nSwitch off")
    link.do_job(switch_off_job)
    print_job_status(switch_off_job)
except zl.ZahnerLinkException as e:
    print(f"\nSwitch-off after disconnect failed: {e}")

connected successfully
ErrorObject: no error occoured
ErrorObject==True: False
ErrorObject code enum: ErrorCodeEnum.NONE
ErrorObject message: no error occoured
[DISCONNECT-THREAD] Waiting 5s before killing the connection ...

Starting EIS measurement (connection will be killed mid-measurement) ...
[DISCONNECT-THREAD] Disabling adapter NOW!
[NET] Disabling 'Ethernet 2' ...
[NET] rc=0 stdout='' stderr=''

caught exception: network connection closed by remote peer
ErrorObject: network connection closed by remote peer
ErrorObject==True: True
ErrorObject code enum: ErrorCodeEnum.CONNECTION_WAS_CLOSED
ErrorObject message: network connection closed by remote peer
job successfull: False
job status: JobStatusEnum.FAILED
job error message: network connection closed by remote peer
JobInfo object id: void
JobInfo object status: JobStatusEnum.FAILED
JobInfo object status detail: JobStatusDetailEnum.CONNECTION_LOSS
JobInfo object error message: network connection closed by remote peer
JobInfo starte